# DEE — Emergencia genuina vía MC: control cruzado T³ vs S³

**Cuatro experimentos integrados** con mismos parámetros de annealing:

| Caso | Topología | F[C] |
|------|-----------|------|
| 1 | T³ (toro periódico) | F = N · var(κ) |
| 2 | T³ | F = N · var(κ) + α · Σ(1−I)² |
| 3 | S³ (esfera, R=0.36) | F = N · var(κ) |
| 4 | S³ | F = N · var(κ) + α · Σ(1−I)² |

**Predicción cuantitativa de DEE (basada en SIM 15):**
- T³ → ⟨κ⟩ → 0, dim_eff → 3
- S³ → ⟨κ⟩ → +0.46, dim_eff → 3

**Si los cuatro casos discriminan correctamente:** validación dinámica del PSG.
**Si los cuatro casos colapsan al mismo ⟨κ⟩:** el modelo no discrimina geometría dinámicamente.

**N = 216** (6³) para mantener tiempo de cómputo bajo control. Tres seeds, cuatro temperaturas.

---

In [ ]:
import os
import numpy as np
from scipy.sparse import csr_matrix, diags
from scipy.spatial import cKDTree
from scipy.linalg import eigh
import matplotlib.pyplot as plt
import time

PLOT_DIR = 'plots_t3_s3'
os.makedirs(PLOT_DIR, exist_ok=True)

def save_plot(name):
    plt.savefig(os.path.join(PLOT_DIR, f'{name}.png'), dpi=120, bbox_inches='tight')
    print(f'  -> {name}.png')

THETA_D = 43.4
N_TARGET = 216  # 6^3
K = 20

# Para que T^3 y S^3 sean comparables en densidad de nodos:
# T^3 con L=1 tiene volumen 1
# S^3 con radio R tiene volumen 2*pi^2 * R^3
# Para igualar volumenes: R = (1/(2*pi^2))^(1/3) ≈ 0.366
L_T3 = 1.0
R_S3 = (1.0 / (2 * np.pi**2))**(1/3)
print(f'L (T³) = {L_T3:.4f}')
print(f'R (S³) = {R_S3:.4f}  → volumen S³ = {2*np.pi**2*R_S3**3:.4f}')
print(f'N = {N_TARGET}, k = {K}')

## Fase A — Funciones core para T³ y S³

In [ ]:
# ============================================================
# Funciones para T^3 (toro periodico)
# ============================================================

def init_uniform_T3(N_target, seed=42, L=1.0):
    rng = np.random.default_rng(seed)
    points = rng.uniform(0, L, size=(N_target, 3))
    return points, N_target

def init_poisson_T3(N_target, seed=137, L=1.0, min_dist=0.05):
    rng = np.random.default_rng(seed)
    points = []
    max_tries = 50 * N_target
    tries = 0
    while len(points) < N_target and tries < max_tries:
        candidate = rng.uniform(0, L, size=3)
        if len(points) == 0:
            points.append(candidate)
        else:
            pts_arr = np.array(points)
            diff = pts_arr - candidate
            diff = diff - L * np.round(diff / L)
            dists = np.linalg.norm(diff, axis=1)
            if dists.min() > min_dist:
                points.append(candidate)
        tries += 1
    return np.array(points), len(points)

def init_grid_perturbed_T3(N_target, seed=271, L=1.0, jitter_factor=0.6):
    rng = np.random.default_rng(seed)
    n_side = round(N_target**(1/3))
    spacing = L / n_side
    coords = np.arange(n_side) * spacing + spacing/2
    gx, gy, gz = np.meshgrid(coords, coords, coords, indexing='ij')
    points = np.stack([gx.ravel(), gy.ravel(), gz.ravel()], axis=1)
    points += rng.uniform(-jitter_factor*spacing, jitter_factor*spacing, size=points.shape)
    points = points % L
    return points, n_side**3

def grid_T3_perfect(N_target, L=1.0, jitter=0.05, seed=999):
    rng = np.random.default_rng(seed)
    n_side = round(N_target**(1/3))
    spacing = L / n_side
    coords = np.arange(n_side) * spacing + spacing/2
    gx, gy, gz = np.meshgrid(coords, coords, coords, indexing='ij')
    points = np.stack([gx.ravel(), gy.ravel(), gz.ravel()], axis=1)
    points += rng.uniform(-jitter*spacing, jitter*spacing, size=points.shape)
    points = points % L
    return points, n_side**3

def neighbors_T3(points, k=20, L=1.0):
    """k-NN con condiciones periodicas. Retorna nbrs, nbr_dists, nbr_image_pos."""
    N = len(points)
    images = []
    image_to_orig = []
    for dx in [-1, 0, 1]:
        for dy in [-1, 0, 1]:
            for dz in [-1, 0, 1]:
                shift = np.array([dx*L, dy*L, dz*L])
                images.append(points + shift)
                image_to_orig.append(np.arange(N))
    images_all = np.concatenate(images, axis=0)
    image_to_orig_all = np.concatenate(image_to_orig)
    tree = cKDTree(images_all)
    
    nbrs = np.zeros((N, k), dtype=int)
    nbr_dists = np.zeros((N, k))
    nbr_image_pos = np.zeros((N, k, 3))
    
    for i in range(N):
        dists, idxs = tree.query(points[i], k=k+1)
        valid_mask = ~((image_to_orig_all[idxs] == i) & (dists < 1e-12))
        valid_idx = np.where(valid_mask)[0][:k]
        nbrs[i] = image_to_orig_all[idxs[valid_idx]]
        nbr_dists[i] = dists[valid_idx]
        nbr_image_pos[i] = images_all[idxs[valid_idx]]
    return nbrs, nbr_dists, nbr_image_pos

def move_T3(points, i, delta, L=1.0):
    """Mover nodo i por delta y wrappear en T^3."""
    new_pts = points.copy()
    new_pts[i] = (points[i] + delta) % L
    return new_pts

def gen_move_T3(rng, sigma):
    """Genera vector de movimiento isotropico en T^3 (R^3 plano)."""
    return rng.normal(0, sigma, size=3)

print('Funciones T³ listas')

In [ ]:
# ============================================================
# Funciones para S^3 (esfera 3-D embebida en R^4)
# ============================================================

def init_uniform_S3(N_target, seed=42, R=R_S3):
    """Puntos uniformemente distribuidos en S^3 via Marsaglia."""
    rng = np.random.default_rng(seed)
    pts = rng.normal(size=(N_target, 4))
    norms = np.linalg.norm(pts, axis=1, keepdims=True)
    pts = pts / norms * R
    return pts, N_target

def init_poisson_S3(N_target, seed=137, R=R_S3, min_dist_factor=0.6):
    """Sampling tipo Poisson sobre S^3 con distancia geodesica minima."""
    rng = np.random.default_rng(seed)
    # min_dist en distancia geodesica (arc length)
    # Espaciado tipico: 2*pi*R / N^(1/3) ≈ ...
    target_spacing = 2 * R * (np.pi**2 * 2 / N_target)**(1/3)
    min_dist = target_spacing * min_dist_factor
    
    points = []
    max_tries = 100 * N_target
    tries = 0
    while len(points) < N_target and tries < max_tries:
        cand = rng.normal(size=4)
        cand = cand / np.linalg.norm(cand) * R
        if len(points) == 0:
            points.append(cand)
        else:
            pts_arr = np.array(points)
            cos_angles = np.clip(np.dot(pts_arr, cand) / R**2, -1, 1)
            geod_dists = R * np.arccos(cos_angles)
            if geod_dists.min() > min_dist:
                points.append(cand)
        tries += 1
    return np.array(points), len(points)

def init_grid_perturbed_S3(N_target, seed=271, R=R_S3):
    """
    'Grid perturbado' sobre S^3: empezar con puntos casi uniformes
    (Fibonacci-like sobre la esfera) y agregar gran perturbacion.
    Equivalente a 'partir de algo casi-ordenado y perturbarlo'.
    """
    rng = np.random.default_rng(seed)
    # Fibonacci-like sampling en S^3 via golden angle generalizado
    pts = []
    phi1 = (np.sqrt(5) - 1) / 2  # golden ratio fraction
    phi2 = phi1**2
    for i in range(N_target):
        s = (i + 0.5) / N_target
        u1 = (i * phi1) % 1
        u2 = (i * phi2) % 1
        # Mapear (s, u1, u2) ∈ [0,1]^3 a S^3
        theta = 2 * np.pi * u1
        phi = 2 * np.pi * u2
        # Coords angulares en S^3: (eta, theta, phi)
        eta = np.arccos(1 - 2*s)  # eta ∈ [0, π]
        x = R * np.array([
            np.cos(eta),
            np.sin(eta) * np.cos(theta),
            np.sin(eta) * np.sin(theta) * np.cos(phi),
            np.sin(eta) * np.sin(theta) * np.sin(phi),
        ])
        pts.append(x)
    pts = np.array(pts)
    
    # Perturbacion grande (similar al jitter del T^3)
    spacing = 2 * R * (np.pi**2 * 2 / N_target)**(1/3)
    perturb = rng.normal(0, spacing*0.4, size=(N_target, 4))
    pts_pert = pts + perturb
    # Reproyectar a S^3
    norms = np.linalg.norm(pts_pert, axis=1, keepdims=True)
    pts_pert = pts_pert / norms * R
    return pts_pert, N_target

def fibonacci_S3(N_target, R=R_S3):
    """S^3 perfecto via Fibonacci-like."""
    pts = []
    phi1 = (np.sqrt(5) - 1) / 2
    phi2 = phi1**2
    for i in range(N_target):
        s = (i + 0.5) / N_target
        u1 = (i * phi1) % 1
        u2 = (i * phi2) % 1
        theta = 2 * np.pi * u1
        phi = 2 * np.pi * u2
        eta = np.arccos(1 - 2*s)
        x = R * np.array([
            np.cos(eta),
            np.sin(eta) * np.cos(theta),
            np.sin(eta) * np.sin(theta) * np.cos(phi),
            np.sin(eta) * np.sin(theta) * np.sin(phi),
        ])
        pts.append(x)
    return np.array(pts), N_target

def neighbors_S3(points, k=20, R=R_S3):
    """k-NN con distancia geodesica en S^3. Retorna nbrs, nbr_dists, nbr_pos."""
    N = len(points)
    # Producto escalar entre todos los pares
    cos_angles = np.clip(points @ points.T / R**2, -1, 1)
    np.fill_diagonal(cos_angles, 1.0 - 1e-12)  # evitar self
    geod_dists = R * np.arccos(cos_angles)
    np.fill_diagonal(geod_dists, np.inf)
    
    nbrs = np.argsort(geod_dists, axis=1)[:, :k]
    nbr_dists = np.take_along_axis(geod_dists, nbrs, axis=1)
    nbr_pos = points[nbrs]  # (N, k, 4)
    return nbrs, nbr_dists, nbr_pos

def move_S3(points, i, delta_4d, R=R_S3):
    """Mover nodo i por delta tangente, reproyectar a S^3."""
    new_pts = points.copy()
    new_pos = points[i] + delta_4d
    new_pos = new_pos / np.linalg.norm(new_pos) * R
    new_pts[i] = new_pos
    return new_pts

def gen_move_S3(rng, sigma, current_pos, R=R_S3):
    """Genera vector tangente a S^3 en la posicion actual.
    1) genera vector aleatorio en R^4
    2) le quita la componente radial
    3) escala a sigma
    """
    v = rng.normal(size=4)
    radial = (v @ current_pos / R**2) * current_pos
    tangent = v - radial
    tan_norm = np.linalg.norm(tangent)
    if tan_norm > 1e-10:
        tangent = tangent / tan_norm
    return tangent * rng.normal(0, sigma)

print('Funciones S³ listas')

In [ ]:
# ============================================================
# Funcional F[C] — universal entre topologias
# ============================================================

def compute_kappa_from_dists(nbr_dists, alpha=2.0):
    """kappa = grado / promedio."""
    deg = np.sum(1.0 / nbr_dists**alpha, axis=1)
    return deg / np.mean(deg)

def compute_isotropy(points, nbr_pos, dim=3):
    """
    Isotropia local I_i = (l_1...l_d)^(1/d) / ((sum l)/d).
    Para T^3: posiciones en R^3, tensor de momento es 3x3.
    Para S^3: posiciones en R^4, pero la 'tangente' al nodo es 3D.
    Calcular el tensor de momento en R^d (donde d = embedding dim).
    Usar solo los 3 mayores autovalores (descartar el cuarto si es S^3).
    """
    N = len(points)
    embed_dim = points.shape[1]
    iso = np.zeros(N)
    for i in range(N):
        diffs = nbr_pos[i] - points[i]  # (k, embed_dim)
        M = (diffs.T @ diffs) / len(diffs)  # embed_dim x embed_dim
        eigvals = np.sort(np.linalg.eigvalsh(M))
        # Tomar los 3 autovalores mas grandes (los de las direcciones tangentes a la variedad 3D)
        eigvals_top = eigvals[-3:]
        eigvals_top = np.maximum(eigvals_top, 1e-12)
        geom_mean = np.exp(np.mean(np.log(eigvals_top)))
        arith_mean = np.mean(eigvals_top)
        iso[i] = geom_mean / arith_mean
    return iso

def compute_F(points, neighbor_fn, alpha_iso=0.0, alpha_kernel=2.0, **kwargs):
    """F = N*var(kappa) + alpha_iso * sum (1-I)^2.
    neighbor_fn: neighbors_T3 o neighbors_S3.
    """
    nbrs, nbr_dists, nbr_pos = neighbor_fn(points, **kwargs)
    kappa = compute_kappa_from_dists(nbr_dists, alpha=alpha_kernel)
    
    N = len(points)
    F_var = N * np.var(kappa)
    
    if alpha_iso > 0:
        iso = compute_isotropy(points, nbr_pos)
        F_iso = np.sum((1.0 - iso)**2)
        F_total = F_var + alpha_iso * F_iso
    else:
        iso = None
        F_iso = 0.0
        F_total = F_var
    
    return F_total, F_var, F_iso, kappa, iso

print('Funciones F universal listas')

In [ ]:
# Calibracion de alpha_iso para cada topologia
# Usamos config aleatoria como referencia

print('Calibrando α_iso:\n')

# T^3
pts_rand_T3, _ = init_uniform_T3(N_TARGET, seed=42, L=L_T3)
_, F_var_T3, F_iso_T3, _, _ = compute_F(
    pts_rand_T3, neighbors_T3, alpha_iso=1.0, k=K, L=L_T3
)
ALPHA_ISO_T3 = F_var_T3 / max(F_iso_T3, 1e-6)
print(f'T³: F_var(rand) = {F_var_T3:.4f}, F_iso(rand) = {F_iso_T3:.4f}')
print(f'    α_iso = {ALPHA_ISO_T3:.4f}')

# S^3
pts_rand_S3, _ = init_uniform_S3(N_TARGET, seed=42, R=R_S3)
_, F_var_S3, F_iso_S3, _, _ = compute_F(
    pts_rand_S3, neighbors_S3, alpha_iso=1.0, k=K, R=R_S3
)
ALPHA_ISO_S3 = F_var_S3 / max(F_iso_S3, 1e-6)
print(f'S³: F_var(rand) = {F_var_S3:.4f}, F_iso(rand) = {F_iso_S3:.4f}')
print(f'    α_iso = {ALPHA_ISO_S3:.4f}')

In [ ]:
# Loop MC universal

def mc_step_universal(points, T, neighbor_fn, move_fn, gen_move_fn,
                       alpha_iso, sigma_move, F_old, rng, **kwargs):
    N = len(points)
    i = rng.integers(N)
    
    if 'R' in kwargs:  # S^3
        delta = gen_move_fn(rng, sigma_move, points[i], R=kwargs['R'])
        points_new = move_fn(points, i, delta, R=kwargs['R'])
    else:  # T^3
        delta = gen_move_fn(rng, sigma_move)
        points_new = move_fn(points, i, delta, L=kwargs['L'])
    
    F_new, _, _, _, _ = compute_F(points_new, neighbor_fn, alpha_iso=alpha_iso, **kwargs)
    delta_F = F_new - F_old
    
    if T > 0:
        accept = (delta_F < 0) or (rng.random() < np.exp(-delta_F / T))
    else:
        accept = (delta_F < 0)
    
    if accept:
        return points_new, F_new, True
    return points, F_old, False

def run_annealing(pts_init, T_schedule, n_steps_per_T,
                  topology, alpha_iso, sigma_move, seed=42, **kwargs):
    """topology: 'T3' o 'S3'."""
    if topology == 'T3':
        neighbor_fn = neighbors_T3
        move_fn = move_T3
        gen_move_fn = gen_move_T3
    else:
        neighbor_fn = neighbors_S3
        move_fn = move_S3
        gen_move_fn = gen_move_S3
    
    rng = np.random.default_rng(seed)
    points = pts_init.copy()
    F_curr, _, _, _, _ = compute_F(points, neighbor_fn, alpha_iso=alpha_iso, **kwargs)
    
    history = {'step': [], 'T': [], 'F': [], 'kappa_var': [], 'iso_mean': []}
    total_step = 0
    
    for T in T_schedule:
        n_acc = 0
        for s in range(n_steps_per_T):
            points, F_curr, acc = mc_step_universal(
                points, T, neighbor_fn, move_fn, gen_move_fn,
                alpha_iso, sigma_move, F_curr, rng, **kwargs
            )
            if acc: n_acc += 1
            
            if total_step % 500 == 0:
                _, _, _, k_, iso_ = compute_F(points, neighbor_fn, alpha_iso=max(alpha_iso,1.0), **kwargs)
                history['step'].append(total_step)
                history['T'].append(T)
                history['F'].append(F_curr)
                history['kappa_var'].append(k_.var())
                history['iso_mean'].append(iso_.mean() if iso_ is not None else 0)
            total_step += 1
        
        ar = n_acc / n_steps_per_T
        print(f'      T={T:6.2f}  acept={ar*100:5.1f}%  F={F_curr:.4f}')
    
    return points, F_curr, history

print('Funciones MC universales listas')

In [ ]:
# Test rapido: que los movimientos de S^3 preservan |x|=R
rng_test = np.random.default_rng(0)
pts_test, _ = init_uniform_S3(N_TARGET)
norms_init = np.linalg.norm(pts_test, axis=1)
print(f'S³ inicial: |x| = {norms_init.mean():.4f} ± {norms_init.std():.6f} (esperado: {R_S3:.4f})')

# Hacer 100 pasos
for _ in range(100):
    delta = gen_move_S3(rng_test, 0.02, pts_test[0])
    pts_test = move_S3(pts_test, 0, delta)
norms_final = np.linalg.norm(pts_test, axis=1)
print(f'S³ tras 100 mov: |x| = {norms_final.mean():.4f} ± {norms_final.std():.6f}')

# Test de neighbors
_, dists_T3, _ = neighbors_T3(pts_rand_T3[:50], k=10, L=L_T3)
_, dists_S3, _ = neighbors_S3(pts_rand_S3[:50], k=10, R=R_S3)
print(f'\nDistancias tipicas:')
print(f'  T³: {dists_T3.mean():.4f} (rango {dists_T3.min():.4f}-{dists_T3.max():.4f})')
print(f'  S³: {dists_S3.mean():.4f} (rango {dists_S3.min():.4f}-{dists_S3.max():.4f})')

---
## Fase B — Configuración del experimento

Cuatro casos a correr. Cada uno con 3 seeds × 4 temperaturas. Casi 1 hora total.

In [ ]:
T_VALUES = {
    'T_alta': THETA_D,
    'T_media': THETA_D / 5,
    'T_baja': THETA_D / 10,
    'T_cero': 0.0
}

# Pasos reducidos (N=216 + 4 casos = mucho)
N_HOT = 1500
N_MID = 1500
N_TARGET_STEPS = 3000

INIT_FNS_T3 = {
    'uniforme': init_uniform_T3,
    'poisson': init_poisson_T3,
    'grid_perturbado': init_grid_perturbed_T3,
}
INIT_FNS_S3 = {
    'uniforme': init_uniform_S3,
    'poisson': init_poisson_S3,
    'grid_perturbado': init_grid_perturbed_S3,
}

def run_one_case(topology, with_isotropy, init_fns, alpha_iso_calibrated, **base_kwargs):
    case_name = f'{topology}_{"iso" if with_isotropy else "varK"}'
    print(f'\n{"="*72}\nCASO: {case_name} (alpha_iso = {alpha_iso_calibrated if with_isotropy else 0:.4f})\n{"="*72}')
    case_results = {}
    alpha_iso_use = alpha_iso_calibrated if with_isotropy else 0.0
    
    for init_name, init_fn in init_fns.items():
        print(f'\n  --- {init_name} ---')
        if topology == 'T3':
            pts_init, N = init_fn(N_TARGET, seed=42 if init_name=='uniforme' else 137 if init_name=='poisson' else 271, L=L_T3)
        else:
            pts_init, N = init_fn(N_TARGET, seed=42 if init_name=='uniforme' else 137 if init_name=='poisson' else 271, R=R_S3)
        
        case_results[init_name] = {'pts_init': pts_init, 'N': N, 'runs': {}}
        
        for T_name, T_target in T_VALUES.items():
            print(f'\n    T={T_target:.2f} ({T_name}):')
            if T_target == 0:
                T_schedule = [THETA_D, THETA_D/3, THETA_D/10, 0.0]
                steps = [N_HOT, N_MID, N_MID, N_TARGET_STEPS]
            else:
                T_schedule = [THETA_D, max(T_target*2, THETA_D/3), T_target]
                steps = [N_HOT, N_MID, N_TARGET_STEPS]
            
            seed_run = 42 + abs(hash(init_name + T_name + case_name)) % 10000
            pts_curr = pts_init.copy()
            full_history = {'step': [], 'T': [], 'F': [], 'kappa_var': [], 'iso_mean': []}
            offset = 0
            t_start = time.time()
            
            for T_phase, n_steps in zip(T_schedule, steps):
                pts_curr, _, hist = run_annealing(
                    pts_curr, [T_phase], n_steps,
                    topology=topology, alpha_iso=alpha_iso_use,
                    sigma_move=0.02, seed=seed_run + offset,
                    **base_kwargs
                )
                for kk in hist:
                    if kk == 'step':
                        full_history[kk].extend([s + offset for s in hist[kk]])
                    else:
                        full_history[kk].extend(hist[kk])
                offset += n_steps
            
            # Mediciones finales
            if topology == 'T3':
                F_f, F_var_f, F_iso_f, kappa_f, iso_f = compute_F(
                    pts_curr, neighbors_T3, alpha_iso=max(alpha_iso_use, 1.0), k=K, L=L_T3
                )
            else:
                F_f, F_var_f, F_iso_f, kappa_f, iso_f = compute_F(
                    pts_curr, neighbors_S3, alpha_iso=max(alpha_iso_use, 1.0), k=K, R=R_S3
                )
            
            case_results[init_name]['runs'][T_name] = {
                'T_target': T_target,
                'pts_final': pts_curr,
                'kappa_final': kappa_f,
                'iso_final': iso_f if iso_f is not None else np.zeros(N),
                'F_final': F_f,
                'history': full_history,
                'time': time.time() - t_start
            }
            print(f'    ✓ var(κ)={kappa_f.var():.4f}, ⟨I⟩={iso_f.mean() if iso_f is not None else 0:.3f}, t={time.time()-t_start:.0f}s')
    return case_results

all_results = {}
print('Setup completo. Ejecutar las celdas de Fase C-F para correr los 4 casos.')

---
## Fase C — Caso 1: T³ con F = N·var(κ)

In [ ]:
all_results['T3_varK'] = run_one_case(
    topology='T3', with_isotropy=False,
    init_fns=INIT_FNS_T3, alpha_iso_calibrated=ALPHA_ISO_T3,
    k=K, L=L_T3
)

---
## Fase D — Caso 2: T³ con F = var(κ) + isotropía

In [ ]:
all_results['T3_iso'] = run_one_case(
    topology='T3', with_isotropy=True,
    init_fns=INIT_FNS_T3, alpha_iso_calibrated=ALPHA_ISO_T3,
    k=K, L=L_T3
)

---
## Fase E — Caso 3: S³ con F = N·var(κ)

In [ ]:
all_results['S3_varK'] = run_one_case(
    topology='S3', with_isotropy=False,
    init_fns=INIT_FNS_S3, alpha_iso_calibrated=ALPHA_ISO_S3,
    k=K, R=R_S3
)

---
## Fase F — Caso 4: S³ con F = var(κ) + isotropía

In [ ]:
all_results['S3_iso'] = run_one_case(
    topology='S3', with_isotropy=True,
    init_fns=INIT_FNS_S3, alpha_iso_calibrated=ALPHA_ISO_S3,
    k=K, R=R_S3
)

---
## Fase G — Análisis comparativo

In [ ]:
def kappa_OR_for_topology(points, k, topology, **kwargs):
    """
    Calcula curvatura de Ollivier-Ricci promedio: indica positividad/negatividad.
    Para esto, usamos el proxy: 1 - <d_vec_to_neighbors_avg> / <d_to_random_at_same_radius>.
    En la practica usamos el Laplaciano discreto.
    
    Mejor: ⟨κ⟩ desde el Laplaciano = trace(L)/N normalizado.
    Para distinguir T³ (κ ≈ 0) vs S³ (κ > 0), usamos kappa_OR_simple:
    para cada arista (i,j), kappa_ij = 1 - W1(N_i, N_j) / d_ij.
    Con aprox W1 ~ |neighborhood_overlap|.
    """
    N = len(points)
    if topology == 'T3':
        nbrs, nbr_dists, _ = neighbors_T3(points, k=k, **kwargs)
    else:
        nbrs, nbr_dists, _ = neighbors_S3(points, k=k, **kwargs)
    
    # Proxy simple: para cada arista (i,j) cercana, calcular fraccion de vecinos comunes
    # kappa_ij = 1 - W1/d_ij ≈ overlap_fraction (positivo si curvatura positiva)
    kappa_avg = 0
    n_pairs = 0
    for i in range(N):
        nbrs_i = set(nbrs[i])
        for jdx in range(min(5, k)):  # solo vecinos cercanos
            j = nbrs[i, jdx]
            nbrs_j = set(nbrs[j])
            overlap = len(nbrs_i & nbrs_j) / len(nbrs_i | nbrs_j)
            kappa_avg += overlap
            n_pairs += 1
    kappa_avg /= max(n_pairs, 1)
    return kappa_avg

# Calcular metricas finales
init_names = ['uniforme', 'poisson', 'grid_perturbado']
T_names_order = ['T_alta', 'T_media', 'T_baja', 'T_cero']
T_array = np.array([T_VALUES[n] for n in T_names_order])
case_names = ['T3_varK', 'T3_iso', 'S3_varK', 'S3_iso']

summary = {}
for case in case_names:
    if case not in all_results:
        print(f'⚠ Falta correr caso: {case}')
        continue
    summary[case] = {'var_kappa': [], 'iso_mean': [], 'kappa_OR': []}
    topo = 'T3' if case.startswith('T3') else 'S3'
    kw = {'L': L_T3} if topo == 'T3' else {'R': R_S3}
    
    for ti, T_name in enumerate(T_names_order):
        var_kappas = []
        iso_means = []
        kappa_ORs = []
        for init_name in init_names:
            r = all_results[case][init_name]['runs'][T_name]
            var_kappas.append(r['kappa_final'].var())
            iso_means.append(r['iso_final'].mean() if r['iso_final'] is not None else 0)
            kappa_ORs.append(kappa_OR_for_topology(r['pts_final'], K, topo, **kw))
        summary[case]['var_kappa'].append(np.mean(var_kappas))
        summary[case]['iso_mean'].append(np.mean(iso_means))
        summary[case]['kappa_OR'].append(np.mean(kappa_ORs))

# Tabla
print('='*78)
print('SÍNTESIS COMPARATIVA — Cuatro casos')
print('='*78)
print()
print(f'{"Caso":12} {"T":>8} {"var(κ)":>10} {"⟨I⟩":>8} {"κ_OR":>8}')
print('-'*60)
for case in case_names:
    if case not in summary: continue
    for ti, T_name in enumerate(T_names_order):
        s = summary[case]
        print(f'{case:12} {T_array[ti]:>8.2f} {s["var_kappa"][ti]:>10.4f} '
              f'{s["iso_mean"][ti]:>8.3f} {s["kappa_OR"][ti]:>8.3f}')
    print()

In [ ]:
# Plots comparativos
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
case_colors = {'T3_varK': 'lightcoral', 'T3_iso': 'crimson',
               'S3_varK': 'lightblue', 'S3_iso': 'steelblue'}
case_labels = {'T3_varK': 'T³ con F=var(κ)', 'T3_iso': 'T³ con F=var(κ)+iso',
               'S3_varK': 'S³ con F=var(κ)', 'S3_iso': 'S³ con F=var(κ)+iso'}

for ax, key, ylabel, title in [
    (axes[0,0], 'var_kappa', 'var(κ)', 'Dispersión de κ'),
    (axes[0,1], 'iso_mean', '⟨I⟩', 'Isotropía local'),
    (axes[1,0], 'kappa_OR', 'κ_OR (proxy curvatura)', 'Curvatura emergente'),
]:
    for case in case_names:
        if case not in summary: continue
        ax.plot(T_array, summary[case][key], 'o-',
                color=case_colors[case], lw=2, markersize=12,
                label=case_labels[case])
    ax.axvline(THETA_D, color='black', linestyle='--', alpha=0.4, label='θ_D')
    ax.set_xlabel('T', fontsize=11)
    ax.set_ylabel(ylabel, fontsize=11)
    ax.set_title(title, fontsize=11)
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)

# Panel 4: discriminacion T3 vs S3
ax = axes[1, 1]
if 'T3_iso' in summary and 'S3_iso' in summary:
    diff = np.array(summary['S3_iso']['kappa_OR']) - np.array(summary['T3_iso']['kappa_OR'])
    ax.plot(T_array, diff, 's-', color='darkgreen', lw=2, markersize=14)
    ax.axhline(0, color='black', alpha=0.5)
    ax.axvline(THETA_D, color='gray', linestyle='--', alpha=0.4)
    ax.set_xlabel('T', fontsize=11)
    ax.set_ylabel('κ_OR(S³) − κ_OR(T³)', fontsize=11)
    ax.set_title('Discriminación dinámica T³ vs S³\n(con isotropía, debería ser >0 a baja T)', fontsize=11)
    ax.grid(alpha=0.3)

plt.tight_layout()
save_plot('110_comparacion_4_casos')
plt.show()

In [ ]:
# Veredicto final
print('='*72)
print('VEREDICTO')
print('='*72)
print()
print('Test crítico: ¿el modelo discrimina T³ vs S³ dinámicamente?')
print('Predicción DEE: κ_OR(S³) > κ_OR(T³) en todas las T, especialmente T baja.')
print()

if 'T3_iso' in summary and 'S3_iso' in summary:
    for ti, T_name in enumerate(T_names_order):
        kappa_T3 = summary['T3_iso']['kappa_OR'][ti]
        kappa_S3 = summary['S3_iso']['kappa_OR'][ti]
        diff = kappa_S3 - kappa_T3
        print(f'  T={T_array[ti]:6.2f}: κ_OR(T³)={kappa_T3:.3f}, κ_OR(S³)={kappa_S3:.3f}, diff={diff:+.3f}')
    
    diff_T_baja = summary['S3_iso']['kappa_OR'][2] - summary['T3_iso']['kappa_OR'][2]
    diff_T_cero = summary['S3_iso']['kappa_OR'][3] - summary['T3_iso']['kappa_OR'][3]
    print()
    if diff_T_cero > 0.05 and diff_T_baja > 0.05:
        print('✓ DISCRIMINACIÓN GEOMÉTRICA DETECTADA')
        print('  El modelo distingue dinámicamente T³ de S³ a baja temperatura.')
    elif diff_T_cero > 0:
        print('~ Discriminación marginal: hay diferencia pero pequeña.')
    else:
        print('✗ Sin discriminación: el modelo no distingue topologías dinámicamente.')

In [ ]:
import shutil
shutil.make_archive('plots_t3_s3', 'zip', PLOT_DIR)
try:
    from google.colab import files
    files.download('plots_t3_s3.zip')
except ImportError:
    pass
print('Listo')